In [ ]:
"""
Sell Bot — Position Monitor -> IBKR Paper Trading -> MongoDB
Polls open trades from MongoDB, checks exit conditions, places limit sells.
Uses avg fill price from broker (or DB entry_price) for accurate P&L.
Run separately from buy_bot.py (different IBKR clientId).
"""

import logging
import sys
import time
import numpy as np
import pandas as pd
from datetime import datetime, timezone
import random

from pymongo import MongoClient
from ib_insync import *

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ─── Logging  (UTF-8 safe: works in Jupyter AND Windows terminal) ─────────────
class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)

    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass


def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh = logging.FileHandler("sell_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("sell_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ─── IBKR Paper Trading ───────────────────────────────────────────────────────
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)

# ─── Trade Rules ──────────────────────────────────────────────────────────────
PROFIT_TARGET    = 0.50
LOSS_LIMIT       = 0.10
MONITOR_INTERVAL = 30

# ─── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client      = MongoClient("mongodb://localhost:27017/")
db                = mongo_client["brkout_database"]
trades_collection = db["trades"]


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Client (Sell-side only)
# ══════════════════════════════════════════════════════════════════════════════
class IBKRSellClient:
    def __init__(self):
        util.startLoop()
        self.ib = IB()

    def connect(self):
        self.ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    # ── Market Data ───────────────────────────────────────────────────────────
    def get_last_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        self.ib.sleep(3)
        self.ib.cancelMktData(contract)
        for label, val in [("last", ticker.last), ("ask", ticker.ask), ("close", ticker.close)]:
            if val and float(val) > 0:
                log.info(f"{symbol}: price source='{label}'  value=${float(val):.4f}")
                return float(val)
        log.warning(f"{symbol}: no price available from IBKR")
        return None

    def get_bars(self, symbol: str, n: int = 70) -> pd.DataFrame | None:
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        for what_to_show in ("TRADES", "MIDPOINT"):
            raw = self.ib.reqHistoricalData(
                contract,
                endDateTime="",
                durationStr="10 D",
                barSizeSetting="5 mins",
                whatToShow=what_to_show,
                useRTH=False,
                keepUpToDate=False,
            )
            if raw:
                log.info(f"{symbol}: got {len(raw)} bars (whatToShow={what_to_show})")
                return pd.DataFrame([
                    {"high": b.high, "low": b.low, "close": b.close}
                    for b in raw
                ]).tail(n).reset_index(drop=True)
        log.warning(f"{symbol}: no bars returned from IBKR")
        return None

    # ── PSAR ──────────────────────────────────────────────────────────────────
    def psar_exit_signal(self, symbol: str) -> bool:
        try:
            df = self.get_bars(symbol, n=70)
            if df is None or len(df) < 10:
                log.warning(f"{symbol}: not enough bars for PSAR, skipping exit signal")
                return False
            psar  = self._compute_psar(df["high"].values, df["low"].values, df["close"].values)
            price = float(df["close"].iloc[-1])
            exit_ = bool(psar[-1] > price)
            log.info(f"{symbol}: PSAR={psar[-1]:.4f}  price={price:.4f}  exit={exit_}")
            return exit_
        except Exception as e:
            log.error(f"{symbol}: PSAR error - {e}")
            return False

    @staticmethod
    def _compute_psar(highs, lows, closes, iaf=0.02, max_af=0.2) -> np.ndarray:
        n    = len(closes)
        psar = closes.copy().astype(float)
        bull = True
        af   = iaf
        hp   = float(highs[0])
        lp   = float(lows[0])
        ep   = float(lows[0])
        for i in range(2, n):
            if bull:
                psar[i] = psar[i-1] + af * (hp - psar[i-1])
                psar[i] = min(psar[i], lows[i-1], lows[i-2])
                if lows[i] < psar[i]:
                    bull = False; psar[i] = hp; lp = lows[i]; ep = lows[i]; af = iaf
                else:
                    if highs[i] > hp:
                        hp = highs[i]; af = min(af + iaf, max_af)
            else:
                psar[i] = psar[i-1] + af * (ep - psar[i-1])
                psar[i] = max(psar[i], highs[i-1], highs[i-2])
                if highs[i] > psar[i]:
                    bull = True; psar[i] = lp; hp = highs[i]; ep = highs[i]; af = iaf
                else:
                    if lows[i] < lp:
                        lp = lows[i]; af = min(af + iaf, max_af)
        return psar

    # ── Avg fill price lookup ─────────────────────────────────────────────────
    def get_avg_fill_price_from_broker(self, symbol: str, order_id: int) -> float | None:
        # Check open/recent trades first via orderStatus
        for trade in self.ib.trades():
            if (
                trade.contract.symbol == symbol
                and trade.order.orderId == order_id
                and trade.orderStatus.avgFillPrice > 0
            ):
                avg = float(trade.orderStatus.avgFillPrice)
                log.info(f"{symbol}: broker avg fill (trades) = ${avg:.4f}")
                return avg

        # FIX: use ib.fills() which returns Fill objects (have .contract + .execution)
        #      ib.executions() returns bare Execution objects with NO .contract attribute
        fills = self.ib.fills()
        log.info(f"{symbol}: scanning {len(fills)} fill(s) for orderId={order_id}")
        matched = [
            f for f in fills
            if f.contract.symbol == symbol and f.execution.orderId == order_id
        ]
        if matched:
            total_qty  = sum(f.execution.shares for f in matched)
            total_cost = sum(f.execution.shares * f.execution.price for f in matched)
            avg = total_cost / total_qty
            log.info(f"{symbol}: fills avg fill = ${avg:.4f}  (qty={total_qty})")
            return avg

        log.warning(f"{symbol}: no fill found for orderId={order_id}")
        return None

    # ── Orders ────────────────────────────────────────────────────────────────
    def place_limit_sell(self, symbol: str, price: float, qty: int):
        contract = Stock(symbol, "SMART", "USD")
        self.ib.qualifyContracts(contract)
        order = LimitOrder("SELL", qty, round(price, 2))
        order.outsideRth = True   # allow sell pre/post market too
        trade = self.ib.placeOrder(contract, order)
        self.ib.sleep(1)
        log.info(f"SELL {symbol} x{qty} @ ${price:.2f}  orderId={trade.order.orderId}  outsideRth=True")
        return trade

    def get_sell_avg_fill(self, symbol: str, sell_order_id: int,
                          fallback_price: float) -> float:
        self.ib.sleep(3)
        avg = self.get_avg_fill_price_from_broker(symbol, sell_order_id)
        if avg:
            return avg
        log.warning(f"{symbol}: sell fill not confirmed yet, using market price ${fallback_price:.4f}")
        return fallback_price


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB Helpers
# ══════════════════════════════════════════════════════════════════════════════
def get_open_trades() -> list[dict]:
    return list(trades_collection.find({"status": "open"}))


def get_effective_entry_price(trade: dict, ibkr: IBKRSellClient) -> float:
    """
    Priority:
      1. avg_fill_price already stored in DB  (set by buy_bot after fill)
      2. Live broker lookup by order_id       (using ib.fills() not ib.executions())
      3. entry_price stored in DB             (fallback)
    """
    if trade.get("avg_fill_price"):
        return float(trade["avg_fill_price"])

    order_id = trade.get("order_id")
    symbol   = trade["symbol"]
    if order_id:
        broker_avg = ibkr.get_avg_fill_price_from_broker(symbol, order_id)
        if broker_avg:
            trades_collection.update_one(
                {"_id": trade["_id"]},
                {"$set": {"avg_fill_price": broker_avg, "entry_price": broker_avg}},
            )
            log.info(f"{symbol}: updated DB with broker avg fill ${broker_avg:.4f}")
            return broker_avg

    log.warning(f"{symbol}: using stored entry_price=${trade['entry_price']:.4f} as fallback")
    return float(trade["entry_price"])


def close_trade(trade: dict, exit_price: float, reason: str):
    symbol      = trade["symbol"]
    entry_price = float(trade.get("avg_fill_price") or trade["entry_price"])
    pnl         = (exit_price - entry_price) / entry_price

    trades_collection.update_one(
        {"_id": trade["_id"]},
        {"$set": {
            "status":      "closed",
            "exit_price":  exit_price,
            "exit_reason": reason,
            "exited_at":   datetime.now(timezone.utc),
            "pnl_pct":     round(pnl * 100, 2),
        }},
    )
    log.info(
        f"trade closed  {symbol}  entry=${entry_price:.4f}  "
        f"exit=${exit_price:.4f}  pnl={pnl*100:.2f}%  reason={reason}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# Position Monitor Loop
# ══════════════════════════════════════════════════════════════════════════════
def monitor_positions(ibkr: IBKRSellClient):
    log.info("Sell bot position monitor started.")
    while True:
        try:
            open_trades = get_open_trades()
            log.info(f"Checking {len(open_trades)} open trade(s)...")

            for trade in open_trades:
                symbol = trade["symbol"]
                qty    = trade["qty"]

                entry_price = get_effective_entry_price(trade, ibkr)

                current_price = ibkr.get_last_price(symbol)
                if not current_price:
                    log.warning(f"{symbol}: no current price, skipping")
                    continue

                pnl    = (current_price - entry_price) / entry_price
                reason = None

                log.info(
                    f"{symbol}: entry=${entry_price:.4f}  now=${current_price:.4f}  "
                    f"pnl={pnl*100:.2f}%"
                )

                if pnl >= PROFIT_TARGET:
                    reason = f"profit_target ({pnl*100:.1f}%)"
                elif pnl <= -LOSS_LIMIT:
                    reason = f"stop_loss ({pnl*100:.1f}%)"
                elif ibkr.psar_exit_signal(symbol):
                    reason = "psar_exit"

                if reason:
                    sell_trade     = ibkr.place_limit_sell(symbol, current_price, qty)
                    sell_order_id  = sell_trade.order.orderId
                    confirmed_exit = ibkr.get_sell_avg_fill(symbol, sell_order_id, current_price)
                    close_trade(trade, confirmed_exit, reason)

        except Exception as e:
            log.error(f"Monitor error: {e}", exc_info=True)

        time.sleep(MONITOR_INTERVAL)


# ══════════════════════════════════════════════════════════════════════════════
# Entry Point
# ══════════════════════════════════════════════════════════════════════════════
def main():
    log.info("=== Sell Bot Starting ===")
    ibkr = IBKRSellClient()
    ibkr.connect()
    try:
        monitor_positions(ibkr)
    except KeyboardInterrupt:
        log.info("Sell bot stopped.")
    finally:
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    main()


2026-03-12 06:43:49,041 [INFO] === Sell Bot Starting ===
2026-03-12 06:43:49,204 [INFO] IBKR connected | accounts: ['DUD087866']
2026-03-12 06:43:49,206 [INFO] Sell bot position monitor started.
2026-03-12 06:43:49,224 [INFO] Checking 0 open trade(s)...
2026-03-12 06:44:19,228 [INFO] Checking 0 open trade(s)...
2026-03-12 06:44:49,230 [INFO] Checking 0 open trade(s)...
2026-03-12 06:45:19,233 [INFO] Checking 0 open trade(s)...
2026-03-12 06:45:49,236 [INFO] Checking 0 open trade(s)...
2026-03-12 06:46:19,239 [INFO] Checking 0 open trade(s)...
2026-03-12 06:46:49,241 [INFO] Checking 0 open trade(s)...
2026-03-12 06:47:19,244 [INFO] Checking 0 open trade(s)...
2026-03-12 06:47:49,247 [INFO] Checking 0 open trade(s)...
2026-03-12 06:48:19,250 [INFO] Checking 0 open trade(s)...
2026-03-12 06:48:49,253 [INFO] Checking 0 open trade(s)...
2026-03-12 06:49:19,256 [INFO] Checking 0 open trade(s)...
2026-03-12 06:49:49,260 [INFO] Checking 0 open trade(s)...
2026-03-12 06:50:19,263 [INFO] Checkin

Error 10349, reqId 14: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=602596427, symbol='SLAI', exchange='SMART', primaryExchange='NYSE', currency='USD', localSymbol='SLAI', tradingClass='SLAI'), order=LimitOrder(orderId=14, clientId=549, action='SELL', totalQuantity=10, lmtPrice=1.1, outsideRth=True), orderStatus=OrderStatus(orderId=14, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 12, 11, 3, 59, 155751, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 12, 11, 3, 59, 161575, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 14: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-12 07:04:00,168 [INFO] SELL SLAI x10 @ $1.10  orderId=14  outsideRth=True
2026-03-12 07:04:03,182 [INFO] SLAI: broker avg fill (trades) = $1.1000
2026-03-12 07:04:03,184 [INFO] trade closed  SLAI  entry=$1.2400  exit=$1.1000  pnl=-11.29%  reason=stop_loss (-11.3%)
2026-03-12 07:04:33,187 [INFO] Checking 0 open trade(s)...
2026-03-12 07:05:03,190 [INFO] Checking 0 open trade(s)...
2026-03-12 07:05:33,193 [INFO] Checking 0 open trade(s)...
2026-03-12 07:06:03,196 [INFO] Checking 0 open trade(s)...
2026-03-12 07:06:33,201 [INFO] Checking 0 open trade(s)...
2026-03-12 07:07:03,203 [INFO] Checking 0 open trade(s)...
2026-03-12 07:07:33,206 [INFO] Checking 0 open trade(s)...
2026-03-12 07:08:03,209 [INFO] Checking 0 open trade(s)...
2026-03-12 07:08:33,212 [INFO] Checking 0 open trade(s)...
2026-03-12 07:09:03,215 [INFO] Checking 0 open trade(s)...
2026-03-12 07:09:33,217 [INFO] Checking 0 open trade(s)...
2026-03-12 07:10:03,220 [INFO] Checking 0 open trade(s)...
2026-03-12 07:10:33

Error 10349, reqId 532: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=853047000, symbol='ATPC', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='ATPC', tradingClass='SCM'), order=LimitOrder(orderId=532, clientId=549, action='SELL', totalQuantity=10, lmtPrice=5.16, outsideRth=True), orderStatus=OrderStatus(orderId=532, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 12, 13, 16, 13, 956787, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 12, 13, 16, 13, 961248, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 532: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-12 09:16:14,962 [INFO] SELL ATPC x10 @ $5.16  orderId=532  outsideRth=True
2026-03-12 09:16:17,977 [INFO] ATPC: scanning 1 fill(s) for orderId=532
2026-03-12 09:16:17,978 [WARNING] ATPC: no fill found for orderId=532
2026-03-12 09:16:17,978 [WARNING] ATPC: sell fill not confirmed yet, using market price $5.1600
2026-03-12 09:16:17,980 [INFO] trade closed  ATPC  entry=$5.6700  exit=$5.1600  pnl=-8.99%  reason=psar_exit
2026-03-12 09:16:21,035 [INFO] HIMX: price source='last'  value=$12.2700
2026-03-12 09:16:21,036 [INFO] HIMX: entry=$12.1300  now=$12.2700  pnl=1.15%
2026-03-12 09:16:21,359 [INFO] HIMX: got 1655 bars (whatToShow=TRADES)
2026-03-12 09:16:21,364 [INFO] HIMX: PSAR=11.1461  price=12.2700  exit=False
2026-03-12 09:16:51,368 [INFO] Checking 1 open trade(s)...
2026-03-12 09:16:54,423 [INFO] HIMX: price source='last'  value=$12.2100
2026-03-12 09:16:54,424 [INFO] HIMX: entry=$12.1300  now=$12.2100  pnl=0.66%
2026-03-12 09:16:54,726 [INFO] HIMX: got 1655 bars (whatToShow=

...

2026-03-12 16:14:20,917 [INFO] SELL ISPC x10 @ $0.42  orderId=2917  outsideRth=True
2026-03-12 16:14:23,928 [INFO] ISPC: broker avg fill (trades) = $0.4205
2026-03-12 16:14:23,935 [INFO] trade closed  ISPC  entry=$0.2755  exit=$0.4205  pnl=52.63%  reason=profit_target (53.2%)
2026-03-12 16:14:53,937 [INFO] Checking 1 open trade(s)...
2026-03-12 16:14:56,998 [INFO] NXGL: price source='last'  value=$1.1500
2026-03-12 16:14:57,000 [INFO] NXGL: entry=$1.2300  now=$1.1500  pnl=-6.50%
2026-03-12 16:14:57,265 [INFO] NXGL: got 1376 bars (whatToShow=TRADES)
2026-03-12 16:14:57,269 [INFO] NXGL: PSAR=0.9481  price=1.1500  exit=False
2026-03-12 16:15:27,273 [INFO] Checking 1 open trade(s)...
2026-03-12 16:15:30,333 [INFO] NXGL: price source='last'  value=$1.1482
2026-03-12 16:15:30,335 [INFO] NXGL: entry=$1.2300  now=$1.1482  pnl=-6.65%
2026-03-12 16:15:30,593 [INFO] NXGL: got 1377 bars (whatToShow=TRADES)
2026-03-12 16:15:30,596 [INFO] NXGL: PSAR=0.9606  price=1.1499  exit=False
2026-03-12 16:16:

Error 10349, reqId 2945: Order TIF was set to GTC based on order preset.
Canceled order: Trade(contract=Stock(conId=532829485, symbol='NXGL', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='NXGL', tradingClass='SCM'), order=LimitOrder(orderId=2945, clientId=549, action='SELL', totalQuantity=10, lmtPrice=1.1, outsideRth=True), orderStatus=OrderStatus(orderId=2945, status='Cancelled', filled=0.0, remaining=0.0, avgFillPrice=0.0, permId=0, parentId=0, lastFillPrice=0.0, clientId=0, whyHeld='', mktCapPrice=0.0), fills=[], log=[TradeLogEntry(time=datetime.datetime(2026, 3, 12, 20, 18, 17, 16904, tzinfo=datetime.timezone.utc), status='PendingSubmit', message='', errorCode=0), TradeLogEntry(time=datetime.datetime(2026, 3, 12, 20, 18, 17, 21948, tzinfo=datetime.timezone.utc), status='Cancelled', message='Error 10349, reqId 2945: Order TIF was set to GTC based on order preset.', errorCode=10349)], advancedError='')


2026-03-12 16:18:18,030 [INFO] SELL NXGL x10 @ $1.10  orderId=2945  outsideRth=True
2026-03-12 16:18:21,046 [INFO] NXGL: broker avg fill (trades) = $1.1100
2026-03-12 16:18:21,050 [INFO] trade closed  NXGL  entry=$1.2300  exit=$1.1100  pnl=-9.76%  reason=stop_loss (-10.6%)
2026-03-12 16:18:51,052 [INFO] Checking 0 open trade(s)...
2026-03-12 16:19:21,055 [INFO] Checking 0 open trade(s)...
2026-03-12 16:19:51,060 [INFO] Checking 0 open trade(s)...
2026-03-12 16:20:21,063 [INFO] Checking 0 open trade(s)...
2026-03-12 16:20:51,067 [INFO] Checking 0 open trade(s)...
2026-03-12 16:21:21,070 [INFO] Checking 0 open trade(s)...
2026-03-12 16:21:51,072 [INFO] Checking 0 open trade(s)...
2026-03-12 16:22:21,075 [INFO] Checking 0 open trade(s)...
2026-03-12 16:22:51,077 [INFO] Checking 0 open trade(s)...
2026-03-12 16:23:21,080 [INFO] Checking 0 open trade(s)...
2026-03-12 16:23:51,083 [INFO] Checking 0 open trade(s)...
2026-03-12 16:24:21,087 [INFO] Checking 0 open trade(s)...
2026-03-12 16:24:5

[WinError 10054] An existing connection was forcibly closed by the remote host


2026-03-13 12:03:30,006 [ERROR] Monitor error: Socket disconnect
Traceback (most recent call last):
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\ib_insync\util.py", line 341, in run
    result = loop.run_until_complete(task)
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\nest_asyncio.py", line 98, in run_until_complete
    return f.result()
           ~~~~~~~~^^
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\futures.py", line 194, in result
    raise self._make_cancelled_error()
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\tasks.py", line 306, in __step_run_and_handle_result
    result = coro.throw(exc)
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\ib_insync\ib.py", line 1798, in qualifyContractsAsync
    detailsLists = await asyncio.gather(
                   ^^^^^^^^^^^^^^^^^^^^^
        *(self.reqContractDetailsAsync(c) for c in contracts))
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
asyncio.exceptions.CancelledError

During handling of the above exce